# Phân tích baseline v8
Planner chọn danh sách tool có thứ tự một lần. Caller chỉ điền arg/value cho slot hiện tại. Summarizer quyết định sang slot tiếp, retry cùng slot sau lỗi hoặc kết luận. Không quay planner, không chèn tool.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path("/kaggle/working/lifemate_baseline_v8"), Path("baseline_v8/outputs"), Path("outputs"), Path("../outputs")]
ROOT = next((p for p in candidates if p.exists()), candidates[0])  # Đổi ROOT nếu report nằm chỗ khác

def jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()] if path.exists() else []

def read_run(mode):
    base = ROOT / mode
    return {
        "summary": json.loads((base/"summary.json").read_text()) if (base/"summary.json").exists() else {},
        "cases": pd.DataFrame(jsonl(base/"traces.jsonl")),
        "metrics": pd.read_csv(base/"cases.csv") if (base/"cases.csv").exists() and (base/"cases.csv").stat().st_size else pd.DataFrame(),
        "status": json.loads((base/"run_status.json").read_text()) if (base/"run_status.json").exists() else {},
    }
runs = {mode: read_run(mode) for mode in ("workflow_gold", "baseline_asr")}
for mode, run in runs.items():
    print(mode, run["status"])
    if run["summary"] and run["summary"].get("architecture") != "ordered_plan_caller_summarizer":
        print("Report thuộc kiến trúc khác:", run["summary"].get("architecture"))
    display(run["summary"])


## Metric và lỗi
`invalid_model_output`: sai contract ở plan/call/summarize, gồm done sớm hoặc next sai trạng thái. Env errors được đưa qua summarizer để chọn retry cùng tool/slot hoặc give_up. `done` chỉ sau khi hết plan, vẫn cần audit đã làm đủ request hay chưa. `environment_no_progress`: read/lỗi env lặp; `duplicate_write`: chặn write trùng. `give_up` có thể do plan sai/thiếu bước hoặc tác động không khắc phục được.


In [ ]:
for mode, run in runs.items():
    df = run["metrics"]
    if df.empty:
        continue
    print(mode)
    display(df[[c for c in ["id", "score_proxy", "state_hash_match", "stop_reason", "tool_count", "plan_step_count", "processed_steps", "skipped_steps", "retry_requests", "tool_order_recall", "planner_calls", "caller_calls", "summarizer_calls", "llm_calls", "agent_latency_s"] if c in df]])
    display(df.stop_reason.value_counts().rename("count").to_frame())
    display(df.loc[df.score_proxy < 1, ["id", "error_flags", "missing_tool_names", "extra_tool_names"]])


## Chi phí ba node
Planner một lượt. Mỗi caller attempt hoặc skip đi qua summarizer, trừ guard. N slot xử lý hết thường cần 1+2N lượt LLM; retry tăng 2 lượt mỗi lần. Plan rỗng vẫn chạy summarizer (2 lượt tổng). Decisions gồm planner event và các caller/summarizer attempts. Trace có tool_plan, step_index, selected_tool, caller_output, step_status, summarizer_output.


In [ ]:
node_rows = []
for mode, run in runs.items():
    for case in run["cases"].to_dict("records"):
        for event in case.get("trace", []):
            for node, output in event.get("node_outputs", {}).items():
                node_rows.append({"mode": mode, "id": case["id"], "decision": event["decision"], "node": node,
                    **{k: output.get(k) for k in ["prompt_tokens", "generation_tokens", "latency_s"]}})
nodes = pd.DataFrame(node_rows)
if not nodes.empty:
    display(nodes.groupby(["mode", "node"])[["prompt_tokens", "generation_tokens", "latency_s"]].agg(["count", "mean", "max"]))


In [ ]:
def show_case(sample_id, mode="workflow_gold"):
    run = runs[mode]
    traces = run["cases"]
    if traces.empty or sample_id not in set(traces.id):
        print("Không tìm thấy", sample_id)
        return
    metric = run["metrics"]
    if not metric.empty:
        display(metric.loc[metric.id == sample_id].T)
    case = traces.loc[traces.id == sample_id].iloc[0].to_dict()
    print(json.dumps(case, ensure_ascii=False, indent=2))

# show_case("<sample_id>")


## Diễn giải
- Kiểm tool_plan đủ lookup/hành động và đúng thứ tự chưa. Phần tử trùng tên là nhiều slot, không bị deduplicate.
- Caller chỉ nhìn schema selected_tool. Kiểm arg/value, đúng namespace/bản ghi và dữ kiện đã quan sát.
- skip phải có căn cứ, không dùng để che thiếu thông tin. step_results là metadata, không phải observation.
- Summarizer chỉ caller sau slot thành công/skip; chỉ retry sau lỗi; done chỉ khi remaining_plan_steps=0.
- Hết plan chưa chứng minh đủ nhiệm vụ: kiểm final_answer và các hành động user yêu cầu. Nếu thiếu bước, phải give_up chứ không replan.
- tool_order_recall là LCS tên tool có tính lặp / số gold calls nghiệp vụ, diagnostic offline. Plan có nhánh bị skip có thể khác gold dù thực thi đúng.
- retry_requests đếm yêu cầu retry đã chấp nhận; tool_failures đếm env.call thất bại. Một retry không đổi step_index.
- final_answer không được đưa lại làm feedback và chưa được proxy/hash chấm.
- Dùng wall time/throughput cho batch; workers=1 cho latency riêng. Chưa có số liệu GPU chứng minh cải thiện.
